# Model Experimentation — Advisor Risk Scoring
Reads the three Advisor 360 source tables, engineers features (with median imputation),
then runs Optuna hyperparameter tuning on a LightGBM classifier. All trials logged to MLflow.

In [0]:
import mlflow
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
from lightgbm import LGBMClassifier
from pyspark.sql import functions as F
from src.config import CFG
from src.utils import get_logger

logger = get_logger("models.train")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(CFG.experiment_name)

In [0]:
LABEL_COL = "at_risk"
ID_COL = "Salesforce_Id__c"

FEATURE_COLS = [
    # Book profile — benchmark ratios
    "avg_advisor_value", "avg_vs_team_ratio", "avg_vs_region_ratio",
    "metrics_below_team", "metrics_below_region",
    "pct_below_team", "pct_below_region",
    # Tool adoption
    "total_advisor_sessions", "tools_adopted_count", "avg_sessions_vs_team",
    # Holdings — portfolio concentration
    "total_aum", "num_funds", "max_fund_pct", "hhi_concentration",
]

In [0]:
# ── Read source tables (latest month) ─────────────────────────────
LATEST_MONTH = (
    spark.table(CFG.fq("book_profile"))
    .select(F.max("Month_End_Date__c").alias("latest"))
    .collect()[0]["latest"]
)

# Book profile → benchmark ratios
bp_features = (
    spark.table(CFG.fq("book_profile"))
    .filter(F.col("Month_End_Date__c") == LATEST_MONTH)
    .withColumn("vs_team",
        F.when(F.col("Team_Avg__c") > 0,
               F.col("Advisor_Value__c") / F.col("Team_Avg__c")).otherwise(0.0))
    .withColumn("vs_region",
        F.when(F.col("Region_Avg__c") > 0,
               F.col("Advisor_Value__c") / F.col("Region_Avg__c")).otherwise(0.0))
    .withColumn("below_team",
        (F.col("Advisor_Value__c") < F.col("Team_Avg__c")).cast("int"))
    .withColumn("below_region",
        (F.col("Advisor_Value__c") < F.col("Region_Avg__c")).cast("int"))
    .groupBy("Salesforce_Id__c")
    .agg(
        F.round(F.avg("Advisor_Value__c"), 2).alias("avg_advisor_value"),
        F.round(F.avg("vs_team"), 2).alias("avg_vs_team_ratio"),
        F.round(F.avg("vs_region"), 2).alias("avg_vs_region_ratio"),
        F.sum("below_team").alias("metrics_below_team"),
        F.sum("below_region").alias("metrics_below_region"),
        F.count("*").alias("_n"),
    )
    .withColumn("pct_below_team", F.round(F.col("metrics_below_team") / F.col("_n"), 2))
    .withColumn("pct_below_region", F.round(F.col("metrics_below_region") / F.col("_n"), 2))
    .drop("_n")
)

# Tools adoption → session aggregates
ta_features = (
    spark.table(CFG.fq("tools_adoption"))
    .filter(F.col("Month_End_Date__c") == LATEST_MONTH)
    .withColumn("is_adopted", (F.col("Advisor_Sessions__c") > 0).cast("int"))
    .withColumn("vs_team_sessions",
        F.when(F.col("Team_Sessions__c") > 0,
               F.col("Advisor_Sessions__c") / F.col("Team_Sessions__c")).otherwise(0.0))
    .groupBy("Salesforce_Id__c")
    .agg(
        F.round(F.sum("Advisor_Sessions__c"), 2).alias("total_advisor_sessions"),
        F.sum("is_adopted").alias("tools_adopted_count"),
        F.round(F.avg("vs_team_sessions"), 2).alias("avg_sessions_vs_team"),
    )
)

# Holdings → portfolio concentration
ah_features = (
    spark.table(CFG.fq("advisor_holdings"))
    .filter(F.col("Month_End_Date__c") == LATEST_MONTH)
    .withColumn("pct_sq", (F.col("Percentage__c") / 100) ** 2)
    .groupBy("Salesforce_Id__c")
    .agg(
        F.round(F.sum("Value__c"), 2).alias("total_aum"),
        F.count("*").alias("num_funds"),
        F.round(F.max("Percentage__c"), 2).alias("max_fund_pct"),
        F.round(F.sum("pct_sq"), 2).alias("hhi_concentration"),
    )
)

# ── Join + synthetic label ──────────────────────────────────────
advisor_features = (
    bp_features
    .join(ta_features, on="Salesforce_Id__c", how="inner")
    .join(ah_features, on="Salesforce_Id__c", how="inner")
    .withColumn("at_risk",
        ((F.col("pct_below_team") > 0.5)
         & (F.col("tools_adopted_count") < 4)
         & (F.col("hhi_concentration") > 0.25)).cast("int"))
)

pdf = advisor_features.toPandas()
logger.info(f"Engineered {len(pdf)} rows from {LATEST_MONTH}  |  at_risk rate: {pdf['at_risk'].mean():.4f}")

In [0]:
X = pdf[FEATURE_COLS]
y = pdf[LABEL_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
logger.info(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}  |  at_risk rate: {y.mean():.4f}")
logger.info(f"Features: {len(FEATURE_COLS)}")

In [0]:
def build_pipeline(params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", LGBMClassifier(**params, random_state=42, verbose=-1)),
    ])


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    pipe = build_pipeline(params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall")

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
        mlflow.log_params(params)
        mlflow.log_metric("cv_recall_mean", scores.mean())
        mlflow.log_metric("cv_recall_std", scores.std())

    return scores.mean()


with mlflow.start_run(run_name="lgbm_optuna_tuning") as parent_run:
    study = optuna.create_study(direction="maximize", study_name="advisor_risk_lgbm")
    study.optimize(objective, n_trials=20, show_progress_bar=True)

    # Refit best pipeline on full training set
    best_params = study.best_params
    best_pipeline = build_pipeline(best_params)
    best_pipeline.fit(X_train, y_train)

    # Evaluate on held-out test set
    y_pred = best_pipeline.predict(X_test)
    y_prob = best_pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "test_f1": f1_score(y_test, y_pred, zero_division=0),
        "test_precision": precision_score(y_test, y_pred, zero_division=0),
        "test_recall": recall_score(y_test, y_pred),
        "test_roc_auc": roc_auc_score(y_test, y_prob),
        "best_cv_recall": study.best_value,
    }

    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})
    mlflow.log_metrics(metrics)

    logger.info(f"Best trial: #{study.best_trial.number} | CV Recall: {study.best_value:.4f}")
    logger.info(f"Best params: {best_params}")
    logger.info(f"Test metrics: {metrics}")
    logger.info(f"Parent Run ID: {parent_run.info.run_id}")

In [0]:
logger.info(f"\n{classification_report(y_test, y_pred, target_names=['healthy', 'at_risk'], zero_division=0)}")

# Top 10 trials
trials_df = study.trials_dataframe()[
    ["number", "value", "params_n_estimators", "params_max_depth",
     "params_learning_rate", "params_num_leaves"]
]
trials_df = (
    trials_df.sort_values("value", ascending=False)
    .head(10)
    .rename(columns={"value": "cv_recall"})
)
display(trials_df)

In [0]:
# Copy this dict into the production training notebook
print("best_params = {")
for k, v in best_params.items():
    print(f"    \"{k}\": {repr(v)},")
print("}")